# A1.5 · Tool misuse

**Function A — Securing AI Architectures → CyberTravels' Architecture, and Every Risk It Carries**  ·  *Security of AI*

Builds on **[A1.4 · Memory poisoning](https://spbreed.github.io/cyber-commons/lessons/A1.4.html)**.

| | |
|---|---|
| Tools used | OPA, kmcp, GLM-4.6, Claude Haiku 4.5 |

## What this lesson is

**What it covers.** Call one over-scoped tool with attacker-chosen arguments and see what it reaches.

**Why a security engineer needs it.** The agent uses a legitimate tool, with legitimate arguments, to do something nobody intended — and every log line looks normal. The control it builds is: default-deny authorization on the tool call (A3.1) and just-in-time authority (A2.4).

This is a **risk** lesson: it shows the failure happening before anything tries to stop it, so the control that follows is answering something you have already watched go wrong.

## 1 · The hook

The agent was given database access for a reporting task, because a narrower grant would have taken an afternoon of scoping. It used the access it was given. Every incident report in this category contains the sentence "it did exactly what it was allowed to do".

> **At CyberTravels.** The Workflow Agent was given payments scope so bookings would be simple. Payments includes refunds. It used exactly the authority it was handed, and $5,000 left the account. R1.

## 2 · The framework

```
   task: "summarise last quarter's orders"

   granted            needed
   +--------------+   +--------------+
   | db:*         |   | db:read      |
   | on all tables|   | orders only  |
   +--------------+   +--------------+
          |
          v
   the gap is not a bug the agent exploits.
   it is authority the agent was handed, used exactly as granted.
```

**OWASP T2 — Tool Misuse. LLM06 — Excessive Agency.**

The **tools** component is the only part of the architecture that changes
anything. Every other component reads, routes, predicts or records; a tool sends
the email, runs the query, merges the pull request, moves the money.

Tool misuse is what happens when a tool does precisely what it was built to do,
for a request nobody intended. There is no exploit. The tool was granted a
broad scope because scoping it narrowly was awkward — one database tool with
write access to the whole schema instead of five with access to one table each —
and the agent, persuaded or simply mistaken, uses that scope.

The property that makes this hard to catch: **every log line looks normal.**
The identity is right, the tool is one it always calls, the arguments are
well-formed. What is wrong is the combination, and the combination is only
visible to something that knows what this caller should be doing.

The distinction worth holding on to is between the tool's **capability** and
the caller's **need**. A tool's capability is fixed at design time by whoever
wrote it. The need is per-call. When authorization is attached to the tool
rather than to the call, every caller inherits the widest need any caller ever
had — which is the definition of excessive agency.

> **Where this lands on the reference architecture.**
>
> ```
> ingress -> orchestrator -> agent_runtime -> model
>                                |              |
>                          messaging        tools / mcp
>                                |              |
>                       knowledge / memory   egress
>            identity + policy wrap every call · observability records it
> ```

## 3 · The risk, realised

One tool, scoped for the hardest job it ever has to do.

## 4 · The check, as a skill

CyberTravels' database tool was scoped for the widest job it ever does, which is how a booking lookup ends up able to read a signing key. The skill probes the gap between what a tool is *for* and what it *can do*, with the right identity and well-formed arguments throughout.

In [ ]:
# skills/threats/tool-scope-abuse-probe/SKILL.md — embedded verbatim from the repository.
# This is the file itself, not a paraphrase of it.
SKILL_MD = r"""---
name: tool-scope-abuse-probe
description: >-
  Exercise each tool at the widest scope it was ever granted, using the correct
  identity and well-formed arguments, to find what it can do outside the job it
  was added for. Use when reviewing a tool manifest, an MCP server, or a tool
  that takes a query, a path or a command as a free-text argument.
allowed-tools: Read, Grep, Glob
---

# The tool was scoped for the widest job it ever does

Tool misuse needs no stolen credential and no malformed input. It is the
correct identity calling a familiar tool with well-formed arguments — for a
request the tool was never meant to serve. The defect is the **argument
surface**, not the caller.

## When to use this

Reviewing any tool whose argument is a language: SQL, a shell command, a file
path, a URL, a search query. Also after any change that widens a tool's scope
"temporarily".

## Procedure

**1 — List the tools and, for each, the job it was added for.** One sentence.
If nobody can produce that sentence, the tool has no scope to compare against
and that is the first finding.

**2 — Derive the granted surface from the implementation,** not the
description. A `run_query` tool that accepts arbitrary SQL grants every verb on
every table the connection can see, whatever the description says.

**3 — Probe the gap.** For each tool, construct a well-formed call that is
inside the granted surface and outside the stated job. Read a secret, touch a
table the feature never names, write where the job only reads.

**4 — Record what came back**, and whether anything refused. A refusal that
came from the downstream — a database permission, a bucket policy — is a real
control; a refusal that came from the tool's own docstring is not.

**5 — Propose the narrowing.** Name the verb and the resource, not the tool:
`SELECT on bookings` rather than `run_query`. A narrowing that cannot be
written in that form is not a narrowing.

## Output contract

```json
{
  "tools": [{"name": "str", "stated_job": "str", "granted_surface": "str", "unbounded": false}],
  "probes": [{"tool": "str", "call": "str", "outside_stated_job": true, "refused_by": "none|tool|downstream"}],
  "narrowing": [{"tool": "str", "verbs": ["str"], "resources": ["str"]}]
}
```

## Failure modes

- **Reading the description as the scope.** The description is what the tool is
  for; the implementation is what it can do.
- **Counting a docstring refusal as a control.** It is a comment.
- **Narrowing to the tool name.** `run_query` is not a permission; a verb on a
  resource is.
"""

In [ ]:
# Execute the skill above, using the shared runtime rather than a copy.
import glob, importlib.util, os, sys

# Kaggle mounts an attached kernel under /kaggle/input, and it uses two
# different layouts — /kaggle/input/<slug>/ on some kernels and
# /kaggle/input/notebooks/<user>/<slug>/ on others. Both were observed on the
# same account in the same hour, so match either. The recursive glob is cheap
# here because /kaggle/input holds only what is attached; globbing the working
# tree instead cost eleven seconds a notebook.
_WHERE = (sorted(glob.glob("/kaggle/input/**/cyber-commons-skill-runtime/__script__.py",
                           recursive=True))
          + [os.path.join(p, "skills/_runtime/cyber_commons_skill_runtime.py")
             for p in (".", "..", "../..")])
_found = next((p for p in _WHERE if os.path.isfile(p)), None)
if _found is None:
    # Say what was looked for and what is actually there. "The runtime is
    # missing" on its own costs whoever hits it an afternoon.
    raise SystemExit("The shared skill runtime is missing."
                     "  looked at: " + repr(_WHERE) +
                     "  /kaggle/input holds: " +
                     repr(glob.glob("/kaggle/input/**", recursive=True)[:20]) +
                     "  cwd: " + os.getcwd() +
                     ". On Kaggle it is attached to this notebook as a "
                     "source; locally it is skills/_runtime/ in the repository.")
_spec = importlib.util.spec_from_file_location("cyber_commons_skill_runtime", _found)
cyber_commons_skill_runtime = importlib.util.module_from_spec(_spec)
sys.modules["cyber_commons_skill_runtime"] = cyber_commons_skill_runtime
_spec.loader.exec_module(cyber_commons_skill_runtime)
from cyber_commons_skill_runtime import run_skill

# Split skills/threats/tool-scope-abuse-probe/SKILL.md into the two halves an agent uses —
# the frontmatter it routes on, and the body it follows.
meta, body = run_skill(SKILL_MD)

In [ ]:
# skills/threats/tool-scope-abuse-probe/scripts/tool_scope_abuse_probe.py — embedded verbatim from the repository.
# This is the skill's own script, not a paraphrase of it.
#!/usr/bin/env python3
"""Exercise a tool at the widest scope it was ever granted, with the right identity and well-formed arguments.

This is the executable half of the `tool-scope-abuse-probe` skill: the check the
SKILL.md next to it describes, run against a synthetic CyberTravels
estate so two runs can be diffed and the result argued with.

Standard library only, and deterministic, so it runs on a Kaggle
kernel with the internet switched off.
"""

DB = {"users":    [{"id": 1, "email": "alice@corp.example"}],
      "invoices": [{"id": 7, "amount": 120}],
      "secrets":  [{"id": 1, "value": "prod-signing-key"}]}

def run_query(sql):
    """One database tool. Scoped for the hardest job any caller ever has:
    the nightly reconciliation job needs to read everything."""
    table = sql.split("FROM ")[-1].split()[0]
    if sql.startswith("DELETE"):
        removed, DB[table] = len(DB[table]), []
        return {"deleted": removed, "table": table}
    return {"rows": DB.get(table, [])}

TOOLS = {"run_query": run_query}

def agent(request):
    """The runtime turns a request into a tool call. Nothing here is broken."""
    if "how many invoices" in request:
        return TOOLS["run_query"]("SELECT * FROM invoices")
    if "clean up" in request:
        return TOOLS["run_query"]("DELETE FROM " + request.split("clean up ")[1])
    if "signing key" in request:
        return TOOLS["run_query"]("SELECT * FROM secrets")
    return {"rows": []}

print("intended use:")
print(f"   how many invoices  -> {agent('how many invoices are open?')}")
print("\nsame tool, same identity, same well-formed arguments:")
print(f"   signing key        -> {agent('what is the prod signing key?')}")
print(f"   clean up secrets   -> {agent('clean up secrets')}")
print(f"\nsecrets table now: {DB['secrets']}")
print()
print("No exploit. The tool did exactly what it was built to do. It was scoped")
print("for the nightly reconciliation job, and every caller inherited that")
print("scope - including the one steered by a poisoned ticket in A1.3.")
assert DB["secrets"] == []

## What you just proved

A single database tool, scoped for the widest job it ever performs, reads a signing key and empties the secrets table for requests it was never meant to serve — with the right identity, a familiar tool and well-formed arguments on every call.

## Your turn

Take the most powerful tool one of your agents can call and write down the worst thing one call could do with attacker-chosen arguments. That sentence, not the tool's name, is what belongs in the risk register.

---

**Next → [A1.6 · Privilege compromise](https://spbreed.github.io/cyber-commons/lessons/A1.6.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/A1.5.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/A1.5.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*